In [15]:
import duckdb
import pandas as pd

df = pd.read_csv("/Users/shiva/Masters/Big_Data/TMDB_all_movies_2.csv")
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2809159,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2517488,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3159388,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1494823,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1555244,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [16]:
FILE_PATH = "/Users/shiva/Masters/Big_Data/TMDB_all_movies_2.csv"

conn = duckdb.connect(FILE_PATH)

# Load CSV into a raw table
# Load CSV into raw table
conn.sql(f"""
    CREATE OR REPLACE TABLE raw_movies AS
    SELECT * FROM read_csv_auto('{FILE_PATH}',
        nullstr=['', 'NA', 'N/A'],
        header=true
    )
""")

# Inspect what you have
conn.sql("DESCRIBE raw_movies").show()
conn.sql("SELECT COUNT(*) AS total FROM raw_movies").show()
conn.sql("SELECT * FROM raw_movies LIMIT 5").show()

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ title                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ vote_average         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ vote_count           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ status               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ release_date         │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ runtime              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ adult                │ BOOLEAN     │ YES     │ NUL

In [23]:
#Data Cleaning

conn.sql("""
    CREATE OR REPLACE TABLE clean_movies AS
    SELECT
        id,
        title,
        TRIM(directors)                                      AS directors,
        TRY_CAST(release_date AS DATE)                      AS release_date,
        YEAR(TRY_CAST(release_date AS DATE))                AS release_year,
        genres,
        keywords,
         
         

        -- Treat 0.0 as missing for budget and revenue
        NULLIF(budget, 0)                                   AS budget,
        NULLIF(revenue, 0)                                  AS revenue,

        -- ROI only when both are valid
        CASE
            WHEN budget > 0 AND revenue > 0
            THEN ROUND((revenue - budget) / budget * 100, 2)
        END                                                 AS roi_pct,

        runtime,
        ROUND(averageRating, 1)                               AS rating,
        NULLIF(numVotes, 0)                               AS vote_count,
        status,
        spoken_languages,
        "cast"                                                AS cast_list

    FROM raw_movies
    WHERE title IS NOT NULL
      AND status = 'Released'
""")

# Verify
conn.sql("""
    SELECT
        COUNT(*)           AS total_rows,
        COUNT(budget)      AS has_budget,
        COUNT(revenue)     AS has_revenue,
        COUNT(rating) AS has_rating,
        COUNT(directors)    AS has_directors
    FROM clean_movies
""").show()

┌────────────┬────────────┬─────────────┬────────────┬───────────────┐
│ total_rows │ has_budget │ has_revenue │ has_rating │ has_directors │
│   int64    │   int64    │    int64    │   int64    │     int64     │
├────────────┼────────────┼─────────────┼────────────┼───────────────┤
│     433935 │      29236 │       16412 │     433935 │        424012 │
└────────────┴────────────┴─────────────┴────────────┴───────────────┘



In [28]:
conn.sql("""
    CREATE OR REPLACE TABLE vfx_movies AS
    SELECT *
    FROM clean_movies
    WHERE (
        LOWER(genres) LIKE '%science fiction%'
        OR LOWER(genres) LIKE '%action%'
        OR LOWER(genres) LIKE '%fantasy%'
        OR LOWER(genres) LIKE '%adventure%'
        OR LOWER(keywords) LIKE '%visual effects%'
        OR LOWER(keywords) LIKE '%cgi%'
        OR LOWER(keywords) LIKE '%vfx%'
        OR LOWER(keywords) LIKE '%special effects%'
        OR LOWER(keywords) LIKE '%computer generated%'
    )
""")

conn.sql("SELECT COUNT(*) AS vfx_movie_count FROM vfx_movies").show()

┌─────────────────┐
│ vfx_movie_count │
│      int64      │
├─────────────────┤
│           60225 │
└─────────────────┘



In [30]:
conn.sql("""
    COPY vfx_movies
    TO '../raw_files/vfx_movies_1.parquet'
    (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

print("VFX Parquet saved!")

VFX Parquet saved!


In [32]:
OUTPUT_DIR = "../raw_files"

# Movie nodes — VFX only
conn.sql(f"""
    COPY (
        SELECT DISTINCT
            id              AS movieId,
            title,
            release_year,
            rating,
            budget,
            revenue,
            roi_pct
        FROM vfx_movies
        WHERE title IS NOT NULL
    )
    TO '{OUTPUT_DIR}/neo4j_movies_1.csv' (HEADER, DELIMITER ',')
""")

# Director nodes — from VFX movies only
conn.sql(f"""
    COPY (
        SELECT DISTINCT
            directors        AS name
        FROM vfx_movies
        WHERE directors IS NOT NULL
          AND TRIM(directors) != ''
    )
    TO '{OUTPUT_DIR}/neo4j_directors_1.csv' (HEADER, DELIMITER ',')
""")

# DIRECTED_BY relationships — VFX movies only
conn.sql(f"""
    COPY (
        SELECT DISTINCT
            id              AS movieId,
            directors        AS directorName
        FROM vfx_movies
        WHERE directors IS NOT NULL
    )
    TO '{OUTPUT_DIR}/neo4j_directed_by_1.csv' (HEADER, DELIMITER ',')
""")

print("Neo4j VFX CSVs ready!")

Neo4j VFX CSVs ready!


In [34]:
# How many rows have NULL director in clean_movies?
conn.sql("""
    SELECT 
        COUNT(*)                                        AS total_movies,
        COUNT(directors)                                 AS has_director,
        COUNT(*) - COUNT(directors)                      AS null_director,
        ROUND(100.0 * COUNT(directors) / COUNT(*), 1)    AS pct_with_director
    FROM clean_movies
""").show()

┌──────────────┬──────────────┬───────────────┬───────────────────┐
│ total_movies │ has_director │ null_director │ pct_with_director │
│    int64     │    int64     │     int64     │      double       │
├──────────────┼──────────────┼───────────────┼───────────────────┤
│       433935 │       424012 │          9923 │              97.7 │
└──────────────┴──────────────┴───────────────┴───────────────────┘

